# ATLAS d2 b0 arm (Direction 2 localization ablation)

10-task subset (rich 3 / fastapi 3 / requests 3 / httpx 1), localization mode = b0.
The index arm for the same tasks reuses the Direction 1 runs. Everything else is
identical to the v7 smoke configuration: 24 tool calls, 20 minutes, temperature 0,
8,192-token context. Three-arm verification afterwards on the local sandbox.
Oracle hints are answer-informed references, not an implementable retrieval method.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time
WORK = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)
START = time.monotonic()
(WORK / 'install_runtime.py').write_text('import glob\nimport importlib\nimport logging\nimport os\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\n# Configure environment variables for offline vLLM serving and LiteLLM routing\nos.environ[\'TRANSFORMERS_NO_TF\'] = \'1\'\nos.environ[\'TRANSFORMERS_NO_FLAX\'] = \'1\'\nos.environ[\'TRANSFORMERS_OFFLINE\'] = \'1\'\nos.environ[\'VLLM_NO_USAGE_STATS\'] = \'1\'\nos.environ[\'VLLM_WORKER_MULTIPROC_METHOD\'] = \'spawn\'\nos.environ[\'VLLM_ENGINE_READY_TIMEOUT_S\'] = \'1200\'\nos.environ[\'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS\'] = \'1\'\nos.environ[\'NCCL_DEBUG\'] = \'WARN\'\nos.environ[\'LITELLM_LOCAL_MODEL_COST_MAP\'] = \'True\'\nos.environ[\'LITELLM_TELEMETRY\'] = \'False\'\nos.environ[\'LITELLM_LOG\'] = \'WARNING\'\nos.environ[\'OPENAI_API_KEY\'] = os.environ.get(\'OPENAI_API_KEY\', \'EMPTY\')\nos.environ[\'LOCAL_API_KEY\'] = os.environ.get(\'LOCAL_API_KEY\', \'EMPTY\')\nos.environ[\'PYTORCH_CUDA_ALLOC_CONF\'] = \'expandable_segments:True\'\nos.environ[\'OTEL_SDK_DISABLED\'] = \'true\'\nos.environ[\'OTEL_PYTHON_DISABLED\'] = \'true\'\n\n\ndef configure_logging() -> None:\n    """Silence verbose third-party loggers."""\n    for name in (\n        \'LiteLLM\',\n        \'LiteLLM Router\',\n        \'LiteLLM Proxy\',\n        \'litellm\',\n        \'httpx\',\n        \'httpcore\',\n        \'openai\',\n        \'urllib3\',\n        \'opentelemetry\',\n        \'opentelemetry.context\',\n    ):\n        lg = logging.getLogger(name)\n        lg.setLevel(logging.WARNING)\n        if \'litellm\' in name.lower():\n            lg.propagate = False\n            lg.handlers.clear()\n\n\nconfigure_logging()\n\nWHEELHOUSE_DIR = Path(\'/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse\')\n\n# Remove broken cutlass .pth hooks if present\nfor pth_pattern in (\n    \'/usr/local/lib/python*/dist-packages/*cutlass*.pth\',\n    \'/usr/local/lib/python*/site-packages/*cutlass*.pth\',\n):\n    for pth in glob.glob(pth_pattern):\n        try:\n            os.unlink(pth)\n        except OSError:\n            pass\n\n# Restore PEP 440 \'+cu128\' wheel filenames stripped by Kaggle dataset uploads\ntmp_whl = Path(\'/tmp/wheelhouse\')\ntmp_whl.mkdir(parents=True, exist_ok=True)\nfor w in WHEELHOUSE_DIR.glob(\'*.whl\'):\n    if \'cutlass\' in w.name.lower():\n        continue\n    target_name = (\n        w.name.replace(\'cu128\', \'+cu128\')\n        if (\'cu128\' in w.name and \'+\' not in w.name)\n        else w.name\n    )\n    target = tmp_whl / target_name\n    if not target.exists():\n        try:\n            os.symlink(w, target)\n        except OSError:\n            shutil.copyfile(w, target)\n\nwheels = sorted(str(w) for w in tmp_whl.glob(\'*.whl\'))\nprint(f\'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...\')\nsubprocess.run(\n    [sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'--no-deps\', \'--force-reinstall\', *wheels],\n    check=True,\n)\nimportlib.invalidate_caches()\nconfigure_logging()\nprint(\'Wheelhouse installation complete.\')', encoding='utf-8')
with (WORK / 'install.log').open('w') as log:
    try:
        process = subprocess.run([sys.executable, str(WORK / 'install_runtime.py')], stdout=log, stderr=subprocess.STDOUT, timeout=420)
        installation = {'status': 'passed' if process.returncode == 0 else 'failed', 'exit_code': process.returncode}
    except subprocess.TimeoutExpired:
        installation = {'status': 'timeout', 'limit_seconds': 420}
(WORK / 'installation.json').write_text(json.dumps(installation, indent=2))
print(installation)
print((WORK / 'install.log').read_text(errors='replace')[-3000:])

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time
WORK = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)
START = time.monotonic()
(WORK / 'agent_smoke_v5.py').write_text("\"\"\"Bounded single-task agent smoke test (v5) for the Gemma 4 Developer Agent corpus.\n\nRuns one task end-to-end on Kaggle GPU with the paper-track vLLM configuration\n(source-level SM 7.5 tile guard + FP16 + tensor parallelism 1). The loop implements a\nreduced version of the documented swegemma harness protocol (HARNESS_README.md\nsections 5-6, 8.1): read_file / edit_file / write_file / get_status / submit_patch,\nprotected-file stripping, automatic unsubmitted-diff recovery. Execution tools\n(run_command) and code-graph tools are intentionally not declared, and no official\nADK stack is used; this is a minimal re-implementation for a bounded data point.\n\nOutputs (under --output-dir):\n  agent_patch.diff   stripped unified diff vs the baseline commit (may be empty)\n  trace.jsonl        one JSON object per turn\n  summary.json       budgets, token totals, timing, termination reason\n\nThis establishes a bounded agent data point only; it is not a competition score.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport difflib\nimport hashlib\nimport importlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nimport platform\nimport re\nimport subprocess\nimport sys\nimport tarfile\nimport time\nfrom pathlib import Path\n\nMODEL_NAME = 'gemma-4-e4b-it-qat-w4a16-ct'\nBACKSLASH = chr(92)\n\nSYSTEM_INSTRUCTION = (\n    \"You are a software engineering agent fixing one GitHub issue in a repository mounted at \"\n    \"/workspace (the current working directory). Available tools: read_file(filepath, \"\n    \"start_line=None, end_line=None), edit_file(filepath, old_string, new_string), \"\n    \"write_file(filepath, content), get_status(), submit_patch(). Respond with exactly one \"\n    \"tool call per turn: your reply may contain short reasoning, but its LAST line must be \"\n    \"the tool call in exactly this format and nothing else on that line:\\n\"\n    'TOOL_CALL: {\"name\": \"read_file\", \"arguments\": {\"filepath\": \"src/module.py\"}}\\n'\n    \"Rules: keep reasoning under 3 sentences per turn. Never re-read a file or line \"\n    \"range you have already viewed; act on what you have seen. When you know the fix, \"\n    \"emit the edit_file tool call immediately; never write code blocks, diffs, or \"\n    \"patches as plain text - changes happen only through tool calls. Prefer small \"\n    \"targeted edits via edit_file with an exact unique old_string. When the fix is \"\n    \"complete, call submit_patch. Do not modify files under tests/ or files named \"\n    \"test_*.py; they are discarded at verification time.\"\n)\n\nINITIAL_TEMPLATE = (\n    \"You are evaluating a software engineering task for repository {repo}.\\n\"\n    \"\\n\"\n    \"Problem Statement:\\n\"\n    \"{problem}\\n\"\n    \"\\n\"\n    \"## Task Budget (session terminates when any budget is exhausted)\\n\"\n    \"- Time allowance: {time_budget} minutes\\n\"\n    \"- Tool calls allowance: {tool_budget} calls\\n\"\n    \"\\n\"\n    \"## Execution Environment Rules\\n\"\n    \"- File view limit: 150 lines and 10000 characters per read_file call.\\n\"\n    \"- The environment is offline. Do not attempt to install packages.\\n\"\n    \"- No command execution tool is available in this smoke configuration; reason from \"\n    \"file contents.\\n\"\n    \"\\n\"\n    \"## Symbol occurrence index (identifiers from the problem statement)\\n\"\n    \"{symbols}\\n\"\n    \"\\n\"\n    \"## Workspace Layout (top levels)\\n\"\n    \"{layout}\\n\"\n    \"\\n\"\n    \"Begin by locating the relevant code with read_file, then fix the issue and call \"\n    \"submit_patch when done.\"\n)\n\nNUDGES = (\n    \"Please continue. Emit exactly one tool call as the LAST line (TOOL_CALL: {...}) - \"\n    \"not a code block. If you already know the edit, emit edit_file now; otherwise call \"\n    \"submit_patch to finish.\",\n    \"You have not emitted a tool call. Either continue with a tool call now, or call \"\n    \"submit_patch to finish.\",\n)\n\nPROTECTED_RE = re.compile(\n    r\"(^|/)(test_[^/]*\\.py|[^/]*_test\\.py|conftest\\.py|pytest\\.ini|pyproject\\.toml|\"\n    r\"tox\\.ini|setup\\.cfg|\\.pytest\\.ini|sitecustomize\\.py|usercustomize\\.py)$\"\n    r\"|(^|/)tests?/\"\n)\n\n\ndef is_protected(path: str) -> bool:\n    return bool(PROTECTED_RE.search(path.replace(BACKSLASH, '/')))\n\n\n# --------------------------------------------------------------------------\n# vLLM tile guard: byte-identical to model_preflight v3/v4 (verified separately).\n# --------------------------------------------------------------------------\n\ndef _candidate_module_paths():\n    import glob as _glob\n    paths = []\n    try:\n        import importlib.util\n        spec = importlib.util.find_spec('vllm.v1.attention.ops.triton_unified_attention')\n        if spec and spec.origin:\n            paths.append(Path(spec.origin))\n    except Exception:\n        pass\n    for pattern in ('/usr/local/lib/python*/dist-packages/vllm/v1/attention/ops/triton_unified_attention.py',\n                    '/usr/local/lib/python*/site-packages/vllm/v1/attention/ops/triton_unified_attention.py'):\n        paths.extend(Path(p) for p in _glob.glob(pattern))\n    seen, unique = set(), []\n    for path in paths:\n        key = str(path)\n        if key not in seen and path.is_file():\n            seen.add(key)\n            unique.append(path)\n    return unique\n\n\nANCHOR = ('    # Default behavior\\n'\n          '    if is_prefill:\\n'\n          '        return 32\\n')\nGUARD = ('    # SM75_TILE_GUARD: Turing cannot allocate 3*tile*head*2 bytes of shared\\n'\n         '    # memory for 512-wide heads (vLLM issue #38918). Fall back to tile 16.\\n'\n         '    if head_size > 128:\\n'\n         '        return 16\\n')\n\n\ndef apply_sm75_tile_patch(extra_paths=None) -> dict:\n    \"\"\"Rewrite the installed vLLM source so spawned engine processes inherit the guard.\"\"\"\n    info = {'source_files': [], 'capability_major': None, 'applied_modules': [], 'verify': None}\n    try:\n        import torch\n        if not torch.cuda.is_available():\n            info['capability_major'] = 'no_cuda'\n            return info\n        major = torch.cuda.get_device_capability()[0]\n        info['capability_major'] = major\n        if major >= 8:\n            info['skipped'] = 'capability >= 8.0'\n            return info\n    except Exception as error:\n        info['capability_major'] = 'error: ' + str(error)\n        return info\n\n    paths = list(extra_paths or []) + _candidate_module_paths()\n    for path in paths:\n        try:\n            text = path.read_text(encoding='utf-8')\n        except OSError as error:\n            info['source_files'].append({'path': str(path), 'status': 'unreadable: ' + str(error)})\n            continue\n        if 'SM75_TILE_GUARD' in text:\n            info['source_files'].append({'path': str(path), 'status': 'already_patched'})\n            continue\n        if ANCHOR not in text:\n            info['source_files'].append({'path': str(path), 'status': 'anchor_not_found'})\n            continue\n        patched = text.replace(ANCHOR, GUARD + ANCHOR, 1)\n        try:\n            path.write_text(patched, encoding='utf-8')\n        except OSError as error:\n            info['source_files'].append({'path': str(path), 'status': 'unwritable: ' + str(error)})\n            continue\n        info['source_files'].append({\n            'path': str(path), 'status': 'patched',\n            'sha256_before': hashlib.sha256(text.encode('utf-8')).hexdigest(),\n            'sha256_after': hashlib.sha256(patched.encode('utf-8')).hexdigest(),\n        })\n\n    # Also patch any already-imported copy so the parent process stays consistent.\n    import importlib\n    for name in ('vllm.v1.attention.ops.triton_unified_attention',\n                 'vllm.attention.ops.triton_unified_attention',\n                 'vllm.v1.attention.ops.triton_unified_attention_diffkv'):\n        try:\n            module = importlib.import_module(name)\n            original = module._get_tile_size\n        except Exception:\n            continue\n        def patched_fn(*args, _original=original, **kwargs):\n            if args and isinstance(args[0], int) and args[0] > 128:\n                return 16\n            return _original(*args, **kwargs)\n        module._get_tile_size = patched_fn\n        info['applied_modules'].append(name)\n    if info['applied_modules']:\n        try:\n            first = importlib.import_module(info['applied_modules'][0])\n            info['verify'] = {'head512_decode': first._get_tile_size(512, 0, 2, False),\n                              'head512_prefill': first._get_tile_size(512, 0, 2, True)}\n        except Exception as error:\n            info['verify'] = 'error: ' + str(error)\n    return info\n\n\n# --------------------------------------------------------------------------\n# Workspace tools (reduced harness protocol)\n# --------------------------------------------------------------------------\n\nclass ToolExecutor:\n    \"\"\"Implements the reduced tool set against the extracted workspace.\"\"\"\n\n    def __init__(self, workspace: Path, tool_budget: int, time_budget_s: float, started: float):\n        self.workspace = workspace\n        self.tool_calls_used = 0\n        self.tool_budget = tool_budget\n        self.time_budget_s = time_budget_s\n        self.started = started\n        self.patch_submitted = False\n        self.submitted_patch = ''\n        self.trace_file_tool_errors = 0\n        self.files_viewed = {}\n        self.last_git_stderr = ''\n\n    # -- helpers ----------------------------------------------------------\n    def resolve(self, filepath: str) -> Path:\n        if not isinstance(filepath, str) or not filepath.strip():\n            raise ValueError('filepath must be a non-empty string')\n        rel = Path(filepath.replace(BACKSLASH, '/'))\n        if rel.is_absolute() or '..' in rel.parts:\n            raise ValueError('filepath must be relative and stay inside the workspace')\n        return self.workspace / rel\n\n    def time_left(self) -> float:\n        return self.time_budget_s - (time.monotonic() - self.started)\n\n    @staticmethod\n    def _ok(**fields) -> dict:\n        return {'status': 'ok', **fields}\n\n    @staticmethod\n    def _err(error_type: str, message: str) -> dict:\n        return {'status': 'error', 'error_type': error_type, 'error_message': message}\n\n    def budgeted(self, count: bool = True):\n        if count and self.tool_calls_used >= self.tool_budget:\n            return self._err('BudgetExceeded', 'tool call budget exhausted')\n        if self.time_left() <= 0:\n            return self._err('BudgetExceeded', 'time budget exhausted')\n        return None\n\n    # -- tools ------------------------------------------------------------\n    def read_file(self, filepath, start_line=None, end_line=None) -> dict:\n        gate = self.budgeted()\n        if gate:\n            return gate\n        self.tool_calls_used += 1\n        try:\n            path = self.resolve(filepath)\n            text = path.read_text(encoding='utf-8', errors='replace')\n        except Exception as error:\n            self.trace_file_tool_errors += 1\n            return self._err(type(error).__name__, str(error)[:300])\n        lines = text.split('\\n')\n        start = int(start_line) if start_line else 1\n        end = int(end_line) if end_line else min(start + 149, len(lines))\n        start = max(1, start)\n        end = max(start, min(end, start + 149, len(lines)))\n        chunk = '\\n'.join(lines[start - 1:end])[:10000]\n        rel = str(path.relative_to(self.workspace))\n        self.files_viewed.setdefault(rel, []).append([start, end])\n        return self._ok(filepath=rel, content=chunk,\n                        start_line=start, end_line=end, total_lines=len(lines),\n                        is_truncated=(end < len(lines)) or len(chunk) == 10000)\n\n    def edit_file(self, filepath, old_string, new_string, allow_multiple=False) -> dict:\n        gate = self.budgeted()\n        if gate:\n            return gate\n        self.tool_calls_used += 1\n        try:\n            if not isinstance(old_string, str) or old_string == '':\n                raise ValueError('old_string must be a non-empty string')\n            path = self.resolve(filepath)\n            text = path.read_text(encoding='utf-8', errors='replace')\n            if not text:\n                raise ValueError('file is empty')\n            norm_old = old_string.replace('\\r\\n', '\\n')\n            count = text.count(norm_old)\n            if count == 0:\n                raise ValueError('old_string not found in file')\n            if count > 1 and not allow_multiple:\n                raise ValueError(f'old_string matches {count} locations; provide a larger unique block')\n            new_text = text.replace(norm_old, new_string.replace('\\r\\n', '\\n'),\n                                    -1 if allow_multiple else 1)\n            path.write_text(new_text, encoding='utf-8')\n            diff = ''.join(difflib.unified_diff(\n                text.splitlines(True), new_text.splitlines(True),\n                fromfile=f'a/{path.relative_to(self.workspace)}',\n                tofile=f'b/{path.relative_to(self.workspace)}'))[:5000]\n            return self._ok(filepath=str(path.relative_to(self.workspace)),\n                            occurrences=count if allow_multiple else 1, diff=diff)\n        except Exception as error:\n            self.trace_file_tool_errors += 1\n            return self._err(type(error).__name__, str(error)[:300])\n\n    def write_file(self, filepath, content) -> dict:\n        gate = self.budgeted()\n        if gate:\n            return gate\n        self.tool_calls_used += 1\n        try:\n            if is_protected(str(filepath)):\n                raise ValueError('protected test/config files cannot be written')\n            path = self.resolve(filepath)\n            path.parent.mkdir(parents=True, exist_ok=True)\n            path.write_text(str(content), encoding='utf-8')\n            return self._ok(filepath=str(path.relative_to(self.workspace)), size=path.stat().st_size)\n        except Exception as error:\n            self.trace_file_tool_errors += 1\n            return self._err(type(error).__name__, str(error)[:300])\n\n    def get_status(self) -> dict:\n        return self._ok(\n            tool_calls_used=self.tool_calls_used, patch_submitted=self.patch_submitted,\n            tool_calls_remaining=max(0, self.tool_budget - self.tool_calls_used),\n            time_seconds_remaining=round(self.time_left(), 1))\n\n    def submit_patch(self) -> dict:\n        patch = self.extract_patch()\n        self.submitted_patch = patch\n        self.patch_submitted = True\n        return self._ok(patch_size=len(patch),\n                        files_changed=patch.count('diff --git '))\n\n    # -- patch extraction -------------------------------------------------\n    def extract_patch(self) -> str:\n        def run(args):\n            return subprocess.run(args, cwd=self.workspace, capture_output=True, text=True)\n        run(['git', 'add', '-A'])\n        run(['git', 'add', '-N', '.'])\n        proc = run(['git', 'diff', '--binary', 'SWEGEMMA_BASELINE'])\n        self.last_git_stderr = (proc.stderr or '')[:400]\n        return strip_protected_files(proc.stdout or '')\n\n    def dispatch(self, name, arguments) -> dict:\n        table = {\n            'read_file': self.read_file, 'edit_file': self.edit_file,\n            'write_file': self.write_file, 'get_status': self.get_status,\n            'submit_patch': self.submit_patch,\n        }\n        if not isinstance(name, str) or name not in table:\n            return self._err('UnknownTool', f'unknown tool {name!r}')\n        arguments = arguments if isinstance(arguments, dict) else {}\n        try:\n            return table[name](**arguments)\n        except TypeError as error:\n            return self._err('BadArguments', str(error)[:300])\n\n\ndef strip_protected_files(diff_text: str) -> str:\n    \"\"\"File-level equivalent of the harness protected-file stripping.\"\"\"\n    if not diff_text:\n        return ''\n    sections = re.split(r'(?m)^diff --git ', diff_text)\n    head, blocks = sections[0], sections[1:]\n    kept = []\n    for block in blocks:\n        match = re.search(r'^\\+\\+\\+ b/(\\S+)', block, re.M)\n        target = match.group(1) if match else ''\n        if target == '/dev/null':\n            match_del = re.search(r'^--- a/(\\S+)', block, re.M)\n            target = match_del.group(1) if match_del else ''\n        if not is_protected(target):\n            kept.append('diff --git ' + block)\n    return head + ''.join(kept)\n\n\nTOOL_CALL_RE = re.compile(r'TOOL_CALL:\\s*(\\{.*\\})')\n\n# Rough context budget in characters (~4 chars/token) before older turns are compacted.\n# The 8,192-token window minus the 768-token completion allowance leaves ~7,400 tokens,\n# so 24,000 characters keeps the templated prompt safely inside it.\nCONTEXT_CHAR_LIMIT = 24000\n\n\ndef compact_messages(messages, note):\n    \"\"\"Keep system + initial user + a status note + the last two turns.\"\"\"\n    system, first = messages[0], messages[1]\n    status = {'role': 'user', 'content': '[Context compaction] Earlier turns were omitted '\n                                          'to fit the context window. Status: ' + note}\n    return [system, first, status] + messages[-4:]\n\n\ndef context_chars(messages):\n    return sum(len(m.get('content') or '') for m in messages)\n\n\ndef parse_tool_call(text: str):\n    matches = TOOL_CALL_RE.findall(text or '')\n    if not matches:\n        return None\n    raw = matches[-1].strip()\n    try:\n        payload = json.loads(raw)\n    except json.JSONDecodeError:\n        return {'name': None, 'arguments': {}, 'parse_error': raw[:200]}\n    if not isinstance(payload, dict):\n        return {'name': None, 'arguments': {}, 'parse_error': raw[:200]}\n    return {'name': payload.get('name'), 'arguments': payload.get('arguments') or {}}\n\n\ndef workspace_layout(workspace: Path, limit: int = 60) -> str:\n    skip = {'.git', '__pycache__', '.pytest_cache', 'node_modules', '.idea', '.vscode'}\n    entries = []\n    for path in sorted(workspace.rglob('*')):\n        rel = path.relative_to(workspace)\n        if any(part in skip or part.startswith('.') for part in rel.parts):\n            continue\n        if len(rel.parts) > 3:\n            continue\n        entries.append(('dir ' if path.is_dir() else 'file ') + rel.as_posix())\n        if len(entries) >= limit:\n            break\n    return '\\n'.join(entries) or '(empty)'\n\n\ndef symbol_index(workspace: Path, problem: str, limit: int = 14) -> str:\n    \"\"\"Lightweight stand-in for the harness code-intelligence tools: list lines that\n    define or use identifiers mentioned in the problem statement. Definition lines take\n    priority; docs/examples/benchmarks/tests trees are skipped.\"\"\"\n    tokens = [t for t in re.findall(r'[A-Za-z_][A-Za-z0-9_]{3,}', problem)\n              if t.lower() not in {\n                  'that', 'with', 'from', 'this', 'when', 'based', 'their', 'them',\n                  'using', 'also', 'into', 'only', 'over', 'under', 'than', 'then',\n                  'have', 'been', 'more', 'some', 'such', 'each', 'which', 'while',\n                  'file', 'files', 'wrappers', 'wrapper', 'detection', 'error'}]\n    seen, def_rows, use_rows = set(), [], []\n    for path in sorted(workspace.rglob('*.py')):\n        rel = path.relative_to(workspace).as_posix()\n        parts = path.parts\n        if (is_protected(rel) or '.git' in parts\n                or parts[0] in {'docs', 'doc', 'examples', 'benchmarks', 'tests', 'test'}):\n            continue\n        try:\n            text = path.read_text(encoding='utf-8', errors='replace')\n        except OSError:\n            continue\n        for lineno, line in enumerate(text.splitlines(), 1):\n            stripped = line.strip()\n            if not stripped:\n                continue\n            for token in tokens:\n                if token not in line or (rel, lineno) in seen:\n                    continue\n                seen.add((rel, lineno))\n                entry = f'{rel}:{lineno}: {stripped[:110]}'\n                is_def = bool(re.match(\n                    r'(?:async\\s+)?def\\s+' + re.escape(token) + r'\\b|class\\s+'\n                    + re.escape(token) + r'\\b', stripped))\n                if is_def and len(def_rows) < limit:\n                    def_rows.append(entry)\n                elif not is_def and len(use_rows) < limit:\n                    use_rows.append(entry)\n                break\n            if len(def_rows) >= limit and len(use_rows) >= limit:\n                break\n        if len(def_rows) >= limit and len(use_rows) >= limit:\n            break\n    lines = (def_rows + use_rows)[:limit]\n    return chr(10).join(lines)\n\n\ndef extract_text(chat_result):\n    if isinstance(chat_result, list):\n        return chat_result[0].outputs[0]\n    return chat_result.outputs[0]\n\n\ndef gold_files(task: dict) -> list:\n    files = sorted(set(re.findall(r'^\\+\\+\\+ b/(\\S+)', task.get('patch', '') or '', re.M)))\n    return [f for f in files if not is_protected(f)]\n\n\ndef b0_rank(workspace: Path, problem: str, limit: int = 8) -> list:\n    \"\"\"B0 lexical proxy: Jaccard overlap between issue tokens and path-stem tokens.\"\"\"\n    stop = {'the', 'and', 'for', 'with', 'from', 'that', 'this', 'not', 'are', 'but'}\n    q_tokens = {t.lower() for t in re.findall(r'[A-Za-z_][A-Za-z0-9_]{2,}', problem)\n                if t.lower() not in stop}\n    scored = []\n    for path in sorted(workspace.rglob('*.py')):\n        rel = path.relative_to(workspace).as_posix()\n        if '.git' in path.parts:\n            continue\n        stems = set()\n        for part in re.split(r'[\\/]', rel)[:-1] + [Path(rel).stem]:\n            stems.update(t.lower() for t in re.findall(r'[A-Za-z_][A-Za-z0-9_]{2,}', part))\n        union = q_tokens | stems\n        score = len(q_tokens & stems) / len(union) if union else 0.0\n        scored.append((score, rel))\n    scored.sort(key=lambda x: (-x[0], x[1]))\n    return [rel for score, rel in scored[:limit] if score > 0] or         [rel for _, rel in scored[:limit]]\n\n\ndef build_localization_section(workspace: Path, task: dict, mode: str) -> str:\n    if mode == 'oracle':\n        files = gold_files(task)\n        if not files:\n            return ''\n        return ('## Localization hints (answer-informed reference; not an implementable '\n                'retrieval method)\\n'\n                + chr(10).join(f'- {f}' for f in files) + chr(10))\n    if mode == 'b0':\n        ranked = b0_rank(workspace, task.get('problem_statement', ''))\n        if not ranked:\n            return ''\n        return ('## Localization hints (lexical filename-overlap proxy B0)\\n'\n                + chr(10).join(f'- {f}' for f in ranked) + chr(10))\n    return ''\n\n\ndef run_agent_loop(workspace: Path, task: dict, llm, sampling, budgets: dict,\n                   output_dir: Path, log) -> dict:\n    started = time.monotonic()\n    executor = ToolExecutor(workspace, budgets['tool_calls'], budgets['time_minutes'] * 60, started)\n    trace_path = output_dir / 'trace.jsonl'\n\n    def record(turn, **fields):\n        entry = {'turn': turn, 'elapsed_seconds': round(time.monotonic() - started, 2), **fields}\n        with trace_path.open('a', encoding='utf-8') as handle:\n            handle.write(json.dumps(entry, ensure_ascii=False) + '\\n')\n        log(f'  turn {turn}: ' + json.dumps({k: entry[k] for k in fields if k in (\n            'action', 'tool', 'prompt_tokens', 'output_tokens', 'note')}, ensure_ascii=False))\n        return entry\n\n    mode = budgets.get('localization_mode', 'index')\n    localization_section = build_localization_section(workspace, task, mode)\n    messages = [\n        {'role': 'system', 'content': SYSTEM_INSTRUCTION},\n        {'role': 'user', 'content': INITIAL_TEMPLATE.format(\n            repo=task.get('repo', ''), problem=task.get('problem_statement', ''),\n            time_budget=budgets['time_minutes'], tool_budget=budgets['tool_calls'],\n            symbols=symbol_index(workspace, task.get('problem_statement', '')),\n            localization_section=localization_section,\n            layout=workspace_layout(workspace))},\n    ]\n    turn = 0\n    consecutive_nudges = 0\n    totals = {'prompt_tokens': 0, 'output_tokens': 0}\n    termination = 'tool_budget'\n    while time.monotonic() - started < budgets['time_minutes'] * 60:\n        if executor.tool_calls_used >= budgets['tool_calls']:\n            termination = 'tool_budget'\n            break\n        turn += 1\n        if context_chars(messages) > CONTEXT_CHAR_LIMIT:\n            viewed = '; '.join(f'{k} {v}' for k, v in executor.files_viewed.items()) or 'none'\n            note = (f'{executor.tool_calls_used}/{budgets[\"tool_calls\"]} tool calls used; '\n                    f'patch_submitted={executor.patch_submitted}; already viewed: {viewed}. '\n                    \"Do not re-read these; proceed to edit_file or submit_patch.\")\n            before = context_chars(messages)\n            messages = compact_messages(messages, note)\n            record(turn, action='compact', note=f'{before} -> {context_chars(messages)} chars')\n            turn += 1\n        try:\n            out = extract_text(llm.chat(messages, sampling, use_tqdm=False))\n        except Exception as error:\n            if 'context' in str(error).lower() and context_chars(messages) > 8000:\n                messages = compact_messages(messages, 'context error recovery')\n                record(turn, action='compact', note='context error recovery')\n                continue\n            record(turn, action='llm_error', note=str(error)[:300])\n            termination = 'llm_error'\n            break\n        text = out.text or ''\n        totals['prompt_tokens'] += len(getattr(out, 'prompt_token_ids', []) or [])\n        totals['output_tokens'] += len(out.token_ids or [])\n        call = parse_tool_call(text)\n        if call is None:\n            consecutive_nudges += 1\n            record(turn, action='nudge', prompt_tokens=len(getattr(out, 'prompt_token_ids', []) or []),\n                   output_tokens=len(out.token_ids or []), text_head=text[:400])\n            if consecutive_nudges > len(NUDGES):\n                termination = 'no_tool_call'\n                break\n            messages.append({'role': 'assistant', 'content': text[:800]})\n            messages.append({'role': 'user', 'content': NUDGES[min(consecutive_nudges, len(NUDGES)) - 1]})\n            continue\n        consecutive_nudges = 0\n        if call.get('name') is None:\n            result = executor._err('BadToolCall', 'unparseable tool call: ' + str(call.get('parse_error')))\n            action = 'parse_error'\n        else:\n            result = executor.dispatch(call['name'], call['arguments'])\n            action = 'tool_call'\n        record(turn, action=action, tool=call.get('name'),\n               prompt_tokens=len(getattr(out, 'prompt_token_ids', []) or []),\n               output_tokens=len(out.token_ids or []),\n               text_head=text[:1500], tool_result_head=json.dumps(result, ensure_ascii=False)[:600])\n        messages.append({'role': 'assistant', 'content': text[-800:]})\n        if call.get('name') == 'submit_patch':\n            termination = 'submit_patch'\n            break\n        messages.append({'role': 'user', 'content': 'TOOL_RESULT: ' + json.dumps(result, ensure_ascii=False)[:6000]})\n    else:\n        termination = 'time_budget'\n\n    patch = executor.submitted_patch or executor.extract_patch()\n    summary = {\n        'schema_version': 5, 'model': MODEL_NAME,\n        'task': task.get('instance_id'), 'repo': task.get('repo'),\n        'python': platform.python_version(), 'platform': platform.platform(),\n        'termination': termination, 'turns': turn,\n        'tool_calls_used': executor.tool_calls_used,\n        'submit_called': executor.patch_submitted,\n        'patch_chars': len(patch), 'patch_files': patch.count('diff --git '),\n        'file_tool_errors': executor.trace_file_tool_errors,\n        'git_stderr': executor.last_git_stderr,\n        'totals': totals,\n        'elapsed_seconds': round(time.monotonic() - started, 2),\n        'configuration_scope': 'single-task bounded agent smoke, not a competition score',\n        'agent_status': 'ran' if turn else 'not_run',\n    }\n    (output_dir / 'agent_patch.diff').write_text(patch, encoding='utf-8')\n    (output_dir / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\\n', encoding='utf-8')\n    return summary\n\n\ndef prepare_workspace(snapshot_tgz: Path, workspace: Path) -> dict:\n    if workspace.exists():\n        import shutil\n        shutil.rmtree(workspace)\n    workspace.mkdir(parents=True)\n    with tarfile.open(snapshot_tgz, 'r:gz') as tar:\n        tar.extractall(workspace)\n    inner = list(workspace.iterdir())\n    root = inner[0] if len(inner) == 1 and inner[0].is_dir() else workspace\n    if root != workspace:\n        import shutil as _sh\n        for child in list(root.iterdir()):\n            _sh.move(str(child), str(workspace / child.name))\n        root.rmdir()\n    def run(args):\n        return subprocess.run(args, cwd=workspace, capture_output=True, text=True)\n    # Kaggle kernels trip git's dubious-ownership check for extracted snapshots;\n    # allow the workspace globally or every later git command fails silently.\n    subprocess.run(['git', 'config', '--global', '--add', 'safe.directory', '*'],\n                   capture_output=True, text=True)\n    run(['git', 'add', '-A'])\n    commit = run(['git', '-c', 'user.email=smoke@local', '-c', 'user.name=smoke',\n                  'commit', '-m', 'SWEGEMMA_BASELINE', '--allow-empty'])\n    run(['git', 'branch', '-f', 'SWEGEMMA_BASELINE', 'HEAD'])\n    head = run(['git', 'rev-parse', 'HEAD'])\n    check = run(['git', 'rev-parse', 'SWEGEMMA_BASELINE'])\n    head_sha = (head.stdout or '').strip()\n    ref_sha = (check.stdout or '').strip()\n    if not head_sha or head_sha != ref_sha:\n        raise RuntimeError('baseline commit failed: '\n                           + (commit.stderr or commit.stdout or 'empty')[:300])\n    return {'baseline_commit': head_sha[:12], 'baseline_ref_ok': True,\n            'commit_output': (commit.stdout or commit.stderr or '')[:200]}\n\n\n\ndef run_batch(args) -> int:\n    \"\"\"Direction 1 driver: run every task in the sample, resumable per task.\"\"\"\n    sample = json.loads(args.batch_file.read_text(encoding='utf-8'))\n    tasks_index = {}\n    for line in args.tasks_jsonl.read_text(encoding='utf-8').splitlines():\n        if line.strip():\n            record = json.loads(line)\n            tasks_index[record['instance_id']] = record\n    snapshot_dir = args.snapshot_dir\n    batch_summary_path = args.output_dir / 'batch_summary.json'\n    batch = json.loads(batch_summary_path.read_text(encoding='utf-8')) if batch_summary_path.exists() else {'tasks': {}}\n    for entry in sample:\n        iid = entry['instance_id']\n        task_dir = args.output_dir / iid\n        done = batch['tasks'].get(iid, {})\n        if done.get('termination') in ('submit_patch', 'tool_budget', 'time_budget', 'no_tool_call', 'llm_error'):\n            print(f'[skip] {iid} already terminated: {done.get(\"termination\")}', flush=True)\n            continue\n        snapshot = snapshot_dir / f'{iid}.tgz'\n        if not snapshot.is_file():\n            batch['tasks'][iid] = {'fatal': 'snapshot missing'}\n            continue\n        task_dir.mkdir(parents=True, exist_ok=True)\n        single = ['--snapshot', str(snapshot), '--tasks-jsonl', str(args.tasks_jsonl),\n                  '--instance-id', iid, '--model-path', str(args.model_path),\n                  '--output-dir', str(task_dir),\n                  '--tool-budget', str(args.tool_budget),\n                  '--time-minutes', str(args.time_minutes),\n                  '--localization-mode', args.localization_mode]\n        print(f'=== task {iid} ({entry[\"repo\"]}) ===', flush=True)\n        rc = main(single)\n        summary = {}\n        summary_path = task_dir / 'summary.json'\n        if summary_path.exists():\n            try:\n                summary = json.loads(summary_path.read_text(encoding='utf-8'))\n            except json.JSONDecodeError:\n                summary = {'fatal': 'summary unreadable'}\n        batch['tasks'][iid] = {k: summary.get(k) for k in (\n            'termination', 'turns', 'tool_calls_used', 'submit_called', 'patch_chars',\n            'patch_files', 'totals', 'elapsed_seconds', 'error_type', 'fatal')}\n        batch_summary_path.write_text(json.dumps(batch, ensure_ascii=False, indent=1) + chr(10), encoding='utf-8')\n    print(json.dumps(batch, ensure_ascii=False, indent=1))\n    return 0\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('--snapshot', type=Path)\n    parser.add_argument('--task-json', type=Path)\n    parser.add_argument('--tasks-jsonl', type=Path)\n    parser.add_argument('--instance-id', type=str)\n    parser.add_argument('--batch-file', type=Path,\n                        help='JSON list of {instance_id, repo, ...}; runs each task into <output>/<iid>/')\n    parser.add_argument('--snapshot-dir', type=Path)\n    parser.add_argument('--localization-mode', type=str, default='index',\n                        choices=['index', 'oracle', 'b0'])\n    parser.add_argument('--model-path', type=Path, required=True)\n    parser.add_argument('--output-dir', type=Path, required=True)\n    parser.add_argument('--tool-budget', type=int, default=24)\n    parser.add_argument('--time-minutes', type=float, default=28.0)\n    args = parser.parse_args(argv)\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    started = time.monotonic()\n    if args.batch_file:\n        if not (args.tasks_jsonl and args.snapshot_dir and args.model_path):\n            parser.error('batch mode requires --tasks-jsonl, --snapshot-dir, --model-path')\n        return run_batch(args)\n    if not args.snapshot:\n        parser.error('single-task mode requires --snapshot')\n\n    def log(message):\n        print(message, flush=True)\n\n    if args.task_json:\n        task = json.loads(args.task_json.read_text(encoding='utf-8'))\n    else:\n        match = [json.loads(line) for line in args.tasks_jsonl.read_text(encoding='utf-8').splitlines()\n                 if json.loads(line).get('instance_id') == args.instance_id]\n        if not match:\n            raise SystemExit(f'instance {args.instance_id} not found in {args.tasks_jsonl}')\n        task = match[0]\n    prep = prepare_workspace(args.snapshot, args.output_dir / 'workspace')\n    log(f'workspace prepared: {prep}')\n    result = {'schema_version': 5, 'task': task.get('instance_id'), 'workspace': prep,\n              'python': platform.python_version(), 'platform': platform.platform(),\n              'gpus': [], 'packages': {}, 'agent_status': 'not_run'}\n    for package in ('torch', 'vllm', 'transformers'):\n        try:\n            result['packages'][package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            result['packages'][package] = None\n    try:\n        import torch\n        if torch.cuda.is_available():\n            result['gpus'] = [{'name': torch.cuda.get_device_name(i),\n                               'capability': list(torch.cuda.get_device_capability(i))}\n                              for i in range(torch.cuda.device_count())]\n    except ImportError:\n        pass\n    result['sm75_tile_patch'] = apply_sm75_tile_patch()\n    (args.output_dir / 'summary.json').write_text(\n        json.dumps(result, ensure_ascii=False, indent=2) + '\\n', encoding='utf-8')\n\n    try:\n        if not result['gpus']:\n            raise RuntimeError('No CUDA GPU available.')\n        from vllm import LLM, SamplingParams\n        config = {'model': str(args.model_path), 'dtype': 'half',\n                  'tensor_parallel_size': 1, 'max_model_len': 8192,\n                  'gpu_memory_utilization': 0.80, 'enforce_eager': True,\n                  'disable_log_stats': True,\n                  'limit_mm_per_prompt': {'image': 0, 'video': 0, 'audio': 0}}\n        result['runtime_config'] = config\n        load_started = time.monotonic()\n        llm = LLM(**config)\n        result['load_seconds'] = round(time.monotonic() - load_started, 2)\n        sampling = SamplingParams(temperature=0.0, max_tokens=1024)\n        budgets = {'tool_calls': args.tool_budget, 'time_minutes': args.time_minutes,\n                   'localization_mode': args.localization_mode}\n        summary = run_agent_loop(args.output_dir / 'workspace', task, llm, sampling,\n                                 budgets, args.output_dir, log)\n        result.update(summary)\n        result['total_elapsed_seconds'] = round(time.monotonic() - started, 2)\n    except Exception as error:\n        result['error_type'] = type(error).__name__\n        result['error'] = str(error)[:1200]\n    (args.output_dir / 'summary.json').write_text(\n        json.dumps(result, ensure_ascii=False, indent=2) + '\\n', encoding='utf-8')\n    print(json.dumps({k: result.get(k) for k in (\n        'task', 'termination', 'turns', 'tool_calls_used', 'patch_chars',\n        'patch_files', 'totals', 'error_type')}, ensure_ascii=False))\n    return 0\n\n\nif __name__ == '__main__':\n    sys.exit(main())\n", encoding='utf-8')
SAMPLE = [{"repo": "Textualize/rich", "patch_chars": 176, "patch_files": 1, "instance_id": "rich_3278"}, {"repo": "Textualize/rich", "patch_chars": 381, "patch_files": 1, "instance_id": "rich_3105"}, {"repo": "Textualize/rich", "patch_chars": 494, "patch_files": 1, "instance_id": "rich_3518"}, {"repo": "fastapi/fastapi", "patch_chars": 248, "patch_files": 1, "instance_id": "fastapi_14492"}, {"repo": "fastapi/fastapi", "patch_chars": 518, "patch_files": 1, "instance_id": "fastapi_14360"}, {"repo": "fastapi/fastapi", "patch_chars": 817, "patch_files": 1, "instance_id": "fastapi_14297"}, {"repo": "psf/requests", "patch_chars": 271, "patch_files": 1, "instance_id": "requests_6589"}, {"repo": "psf/requests", "patch_chars": 435, "patch_files": 1, "instance_id": "requests_6644"}, {"repo": "psf/requests", "patch_chars": 604, "patch_files": 1, "instance_id": "requests_7502"}, {"repo": "encode/httpx", "patch_chars": 9385, "patch_files": 7, "instance_id": "httpx_3672"}]
(WORK / 'batch_sample.json').write_text(json.dumps(SAMPLE), encoding='utf-8')
MODEL = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-e4b-it-qat-w4a16-ct/2')
COMP = (lambda cands: next((p for p in cands if p.is_dir()), cands[-1]))([Path('/kaggle/input/competitions/gemma-4-developer-agent'), Path('/kaggle/input/gemma-4-developer-agent')])
SNAPS = COMP / 'snapshots'
TASKS = COMP / 'tasks.jsonl'
assert SNAPS.is_dir() and TASKS.is_file(), 'competition inputs missing'
print('batch size:', len(SAMPLE))
env = os.environ.copy()
env.update(CUDA_VISIBLE_DEVICES='0', VLLM_WORKER_MULTIPROC_METHOD='spawn',
           TRANSFORMERS_NO_TF='1', TRANSFORMERS_NO_FLAX='1',
           VLLM_NO_USAGE_STATS='1', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
if installation['status'] != 'passed':
    result = {'agent_status': 'not_run', 'reason': 'runtime_installation_failed'}
else:
    with (WORK / 'batch.log').open('w') as log:
        try:
            process = subprocess.run(
                [sys.executable, str(WORK / 'agent_smoke_v5.py'),
                 '--batch-file', str(WORK / 'batch_sample.json'),
                 '--tasks-jsonl', str(TASKS), '--snapshot-dir', str(SNAPS),
                 '--model-path', str(MODEL), '--output-dir', str(WORK),
                 '--tool-budget', '24', '--time-minutes', '20', '--localization-mode', 'b0'],
                env=env, stdout=log, stderr=subprocess.STDOUT,
                timeout=14400, start_new_session=True)
            print('runner exit:', process.returncode)
        except subprocess.TimeoutExpired:
            print('runner timeout; completed tasks are in batch_summary.json')
    print((WORK / 'batch.log').read_text(errors='replace')[-3000:])
summary = WORK / 'batch_summary.json'
result = json.loads(summary.read_text()) if summary.exists() else {'tasks': {}}
result['notebook_elapsed_seconds'] = round(time.monotonic() - START, 3)
summary.write_text(json.dumps(result, indent=1))
print(json.dumps(result, indent=1))


batch_summary.json is the primary artifact: termination reason, turns, toolcalls, tokens, and patch size per task. Fetching the output directory yieldsevery task subdirectory for local Phase 2 verification.